# Submitting a RayJob which lifecycles its own RayCluster

In this notebook, we will go through the basics of using the SDK to:
 * Define a RayCluster configuration
 * Use this configuration alongside a RayJob definition
 * Submit the RayJob, and allow Kuberay Operator to lifecycle the RayCluster for the RayJob

## Defining and Submitting the RayJob

First, we'll need to import the relevant CodeFlare SDK packages. You can do this by executing the below cell.

In [ ]:
from codeflare_sdk import RayJob, ManagedClusterConfig, TokenAuthentication, analyze_queue_utilization

Execute the below cell to authenticate the notebook via OpenShift.

**TODO: Add guide to authenticate locally.**

In [ ]:
auth = TokenAuthentication(
    token = "XXXXX",
    server = "XXXXX",
    skip_tls=False
)
auth.login()

## Checking Queue Utilization

**💡 Pro Tip**: Before creating a lifecycled cluster, let's check the current queue utilization to ensure we're not overloading the system and to choose the most appropriate resources.


In [ ]:
# Check current queue utilization before creating lifecycled cluster
print("=== Checking Queue Utilization ===")

# Use the native SDK function for comprehensive analysis
utilization_analysis = analyze_queue_utilization()

print(f"\n📊 Quick Analysis:")
print(f"  Primary recommendation: {utilization_analysis['recommendations']['primary_suggestion']}")

if utilization_analysis['recommendations']['warnings']:
    print(f"\n⚠️ Warnings:")
    for warning in utilization_analysis['recommendations']['warnings']:
        print(f"  {warning}")

print(f"\n📋 Queue Status:")
for queue in utilization_analysis['queues']:
    print(f"  {queue['name']}: {queue['total']} workloads ({queue['utilization_level']})")
    print(f"    {queue['recommendation']}")

print("\n💡 Based on this analysis, we can make informed decisions about our cluster configuration.")


## Defining the ManagedClusterConfig

Next we'll define the ManagedClusterConfig. Kuberay will use this to spin up a short-lived RayCluster that will only exist as long as the job. Based on the queue utilization analysis above, we can make informed decisions about resource allocation.

In [ ]:
# Define cluster configuration based on queue utilization analysis
cluster_config = ManagedClusterConfig(
    num_workers=2,
    worker_cpu_requests=1,
    worker_cpu_limits=1,
    worker_memory_requests=4,
    worker_memory_limits=4,
    head_accelerators={'nvidia.com/gpu': 0},
    worker_accelerators={'nvidia.com/gpu': 0},
)

## Creating and Submitting the RayJob

Now we can pass the ManagedClusterConfig into the RayJob and submit it. The cluster will be created automatically by Kuberay based on our configuration and the queue utilization we analyzed earlier. You do not need to worry about tearing down the cluster when the job has completed, that is handled for you!

In [ ]:
# Create RayJob with lifecycled cluster based on queue utilization analysis
job = RayJob(
    job_name="demo-rayjob",
    entrypoint="python -c 'print(\"Hello from RayJob!\")'",
    cluster_config=cluster_config,
    namespace="your-namespace"
)

job.submit()

We can check the status of our cluster by executing the below cell. If it's not up immediately, run the cell a few more times until you see that it's in a 'running' state.

In [ ]:
job.status()